# ADP PM Notebook 1 — LLM API Calls and Parameters

**Purpose:** Give product managers a short, practical view of how an LLM API call works in Amazon Bedrock and why parameters change the response.

**Retained from original:** Bedrock setup, minimal LLM call, reusable helper, parameter comparison, system prompt comparison, simple observability log.

In [ ]:
# Uncomment only if your SageMaker image is missing packages.
# %pip install -q boto3 pandas


In [ ]:
import os
import json
import time
import boto3
import pandas as pd
from IPython.display import display

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
BEDROCK_LLM_MODEL_ID = "amazon.nova-lite-v1:0"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

print("AWS Region:", AWS_REGION)
print("LLM model:", BEDROCK_LLM_MODEL_ID)


## 1. Minimal Bedrock Converse call

This is the smallest pattern PMs need to understand: system instruction + user request + parameters → model response.


In [ ]:
response = bedrock_runtime.converse(
    modelId=BEDROCK_LLM_MODEL_ID,
    system=[{"text": "You are a concise AI assistant for product managers."}],
    messages=[
        {"role": "user", "content": [{"text": "Explain an LLM API call in two lines."}]}
    ],
    inferenceConfig={
        "temperature": 0.2,
        "topP": 0.9,
        "maxTokens": 120,
    },
)

print(response["output"]["message"]["content"][0]["text"])
print("Usage:", response.get("usage", {}))


## 2. Reusable helper function

The rest of the notebook uses one helper so that learners can focus on behavior, not boilerplate.


In [ ]:
def ask_llm(
    user_prompt: str,
    system_prompt: str = "You are a concise enterprise AI assistant.",
    temperature: float = 0.2,
    top_p: float = 0.9,
    max_tokens: int = 180,
) -> dict:
    start = time.time()
    response = bedrock_runtime.converse(
        modelId=BEDROCK_LLM_MODEL_ID,
        system=[{"text": system_prompt}],
        messages=[{"role": "user", "content": [{"text": user_prompt}]}],
        inferenceConfig={"temperature": temperature, "topP": top_p, "maxTokens": max_tokens},
    )
    latency_ms = round((time.time() - start) * 1000, 2)
    answer = response["output"]["message"]["content"][0]["text"]
    usage = response.get("usage", {})
    return {
        "prompt": user_prompt,
        "system_prompt": system_prompt,
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "answer": answer,
        "input_tokens": usage.get("inputTokens"),
        "output_tokens": usage.get("outputTokens"),
        "latency_ms": latency_ms,
    }


## 3. Parameter comparison

Use one PM task and change only the parameters.


In [ ]:
pm_task = "Draft a short product-manager summary for using GenAI to improve HR policy search."

experiments = [
    {"temperature": 0.0, "top_p": 0.8, "max_tokens": 120},
    {"temperature": 0.3, "top_p": 0.9, "max_tokens": 120},
    {"temperature": 0.8, "top_p": 0.95, "max_tokens": 120},
    {"temperature": 0.3, "top_p": 0.9, "max_tokens": 60},
]

results = [ask_llm(pm_task, **params) for params in experiments]
results_df = pd.DataFrame(results)
display(results_df[["temperature", "top_p", "max_tokens", "input_tokens", "output_tokens", "latency_ms", "answer"]])


## 4. System prompt comparison

The system prompt is where product behavior, tone, constraints, and safety expectations are encoded.


In [ ]:
user_prompt = "Improve this backlog item: 'AI should help users find payroll policy answers.'"

systems = [
    "You are a helpful general assistant.",
    "You are an ADP product-management assistant. Improve backlog items with clear user value, acceptance criteria, risks, and governance notes.",
]

system_results = [ask_llm(user_prompt, system_prompt=s, temperature=0.2, max_tokens=220) for s in systems]
system_df = pd.DataFrame(system_results)
display(system_df[["system_prompt", "answer", "output_tokens", "latency_ms"]])


## 5. PM reflection checkpoint

Discuss before moving to RAG:

- Which parameters should a PM specify in requirements?
- Which parameters should be hidden from users?
- What should be logged for auditability?
- Why is system prompt design a product decision, not only an engineering decision?
